# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [14]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [15]:
import pandas as pd
from pathlib import Path

In [16]:
DATASET_NAME = "crc"
N_DEG = 50
CSV_PATH = f"../results/loo_summary_{DATASET_NAME}_DEG_{N_DEG}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse
0,crc_120,baseline,Endothelial_CRC,50,0.548619,0.497170,0.10,1.000000,0.10,0.547060,13.249699,15.278936,5856.333946
1,crc_120,baseline,Epithelial_CRC,50,0.442017,0.330887,0.26,0.769231,0.20,0.895650,9.924456,8.871777,652016.069078
2,crc_120,baseline,Fibroblast_CRC,50,0.378247,0.385970,0.16,1.000000,0.16,0.605086,8.445271,10.781323,180004.851410
3,crc_120,baseline,Myeloid_CRC,50,0.510492,0.456200,0.14,1.000000,0.14,0.502754,14.798410,16.269466,36416.168085
4,crc_120,baseline,T_cell_CRC,50,0.836351,0.712031,0.12,1.000000,0.12,0.797638,16.806622,18.496396,22608.232002


In [17]:
# Rename holdout_celltype by replacing the last '_' in with '-'
if DATASET_NAME == "merfish":
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)
else:
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("T_cell", "T-cell", regex=False)

In [18]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse,perturbation
0,crc_120,baseline,Endothelial,50,0.548619,0.497170,0.10,1.000000,0.10,0.547060,13.249699,15.278936,5856.333946,CRC
1,crc_120,baseline,Epithelial,50,0.442017,0.330887,0.26,0.769231,0.20,0.895650,9.924456,8.871777,652016.069078,CRC
2,crc_120,baseline,Fibroblast,50,0.378247,0.385970,0.16,1.000000,0.16,0.605086,8.445271,10.781323,180004.851410,CRC
3,crc_120,baseline,Myeloid,50,0.510492,0.456200,0.14,1.000000,0.14,0.502754,14.798410,16.269466,36416.168085,CRC
4,crc_120,baseline,T-cell,50,0.836351,0.712031,0.12,1.000000,0.12,0.797638,16.806622,18.496396,22608.232002,CRC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,crc_120,spatialprop-pert,Endothelial,50,0.552941,0.530705,0.02,1.000000,0.02,0.313288,8.139927,10.611496,45608.582000,CRC
286,crc_120,spatialprop-pert,Epithelial,50,0.332149,0.175848,0.04,1.000000,0.04,0.247886,6.466678,9.333009,471183.800000,CRC
287,crc_120,spatialprop-pert,Fibroblast,50,0.159184,0.015947,0.00,0.000000,0.00,0.720367,5.674491,9.491432,242850.530000,CRC
288,crc_120,spatialprop-pert,Myeloid,50,0.463433,0.467000,0.06,1.000000,0.06,0.435887,11.597181,15.426000,122815.600000,CRC


In [19]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    #"spearman": +1,
    "pearson": +1,
    "precision": +1,
    #"direction_match_k": +1,
    "edistance_local": -1,
    #"edistance_global": -1,
    #"rmse": -1,
    #"avg_rank": -1,
}

PRETTY_METRIC = {
    #"spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "precision": r"Precision $\uparrow$",
    #"direction_match_k": r"Direction Match (K) $\uparrow$",
    "edistance_local": r"E-dist (local) $\downarrow$",
    #"edistance_global": r"E-dist (global) $\downarrow$",
    #"rmse": r"RMSE $\downarrow$",
    #"avg_rank": r"Avg. Rank $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    "spatialprop": "SpatialProp",
    "mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    "cellina-graph": "cellina-GAT",
    "cellina": "cellina",
    "cellina-pert": r"cellina$_{node-pert=200}$",
    "spatialprop-pert": r"SpatialProp$_{node-pert=200}$"
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    "SpatialProp",
    "MintFlow",
    "cellina (ablated)",
    "cellina-GAT",
    "cellina",
    r"cellina$_{node-pert=200}$",
    r"SpatialProp$_{node-pert=200}$",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'cellina-GAT', 'CPA',
       'scGen', 'MintFlow', 'SpatialProp', 'cellina$_{node-pert=200}$',
       'SpatialProp$_{node-pert=200}$'], dtype=object)

In [20]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)

In [21]:
agg

pearson           precision  \
                                                mean       std      mean   
perturbation model_name                                                    
CRC          mean shift                     0.507617  0.254893  0.180000   
             CPA                            0.680111  0.191169  0.223333   
             scGen                          0.499778  0.384345  0.216000   
             SpatialProp                    0.384039  0.343696  0.082000   
             MintFlow                       0.658650  0.318159  0.257600   
             cellina (ablated)              0.721768  0.242297  0.293333   
             cellina-GAT                    0.886137  0.093179  0.408667   
             cellina                        0.847046  0.157098  0.369333   
             cellina$_{node-pert=200}$      0.609720  0.302086  0.272667   
             SpatialProp$_{node-pert=200}$  0.364864  0.343329  0.056000   

                                                     edistance_local            
                                                 std            mean       std  
perturbation model_name                                                         
CRC          mean shift                     0.128439       15.504938  4.519661  
             CPA                            0.187770        2.683677  2.108607  
             scGen                          0.202731        2.600304  1.742439  
             SpatialProp                    0.151712        9.446245  4.173289  
             MintFlow                       0.213020       15.132276  2.061794  
             cellina (ablated)              0.202831        2.744164  2.217761  
             cellina-GAT                    0.201284        2.818967  2.101087  
             cellina                        0.197273        3.031160  2.818150  
             cellina$_{node-pert=200}$      0.225693       13.696438  2.596384  
             SpatialProp$_{node-pert=200}$  0.088731       10.478952  4.534562

In [22]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.3f} $\\pm$ {std:.3f}" if not pd.isna(std) else f"{mean:.3f}"
    return f"\\textbf{{{s}}}" if bold else s

In [23]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Pearson $\uparrow$  \
perturbation model_name                                                  
CRC          mean shift                              0.508 $\pm$ 0.255   
             CPA                                     0.680 $\pm$ 0.191   
             scGen                                   0.500 $\pm$ 0.384   
             SpatialProp                             0.384 $\pm$ 0.344   
             MintFlow                                0.659 $\pm$ 0.318   
             cellina (ablated)                       0.722 $\pm$ 0.242   
             cellina-GAT                    \textbf{0.886 $\pm$ 0.093}   
             cellina                                 0.847 $\pm$ 0.157   
             cellina$_{node-pert=200}$               0.610 $\pm$ 0.302   
             SpatialProp$_{node-pert=200}$           0.365 $\pm$ 0.343   

                                                  Precision $\uparrow$  \
perturbation model_name                                                  
CRC          mean shift                              0.180 $\pm$ 0.128   
             CPA                                     0.223 $\pm$ 0.188   
             scGen                                   0.216 $\pm$ 0.203   
             SpatialProp                             0.082 $\pm$ 0.152   
             MintFlow                                0.258 $\pm$ 0.213   
             cellina (ablated)                       0.293 $\pm$ 0.203   
             cellina-GAT                    \textbf{0.409 $\pm$ 0.201}   
             cellina                                 0.369 $\pm$ 0.197   
             cellina$_{node-pert=200}$               0.273 $\pm$ 0.226   
             SpatialProp$_{node-pert=200}$           0.056 $\pm$ 0.089   

                                           E-dist (local) $\downarrow$  
perturbation model_name                                                 
CRC          mean shift                             15.505 $\pm$ 4.520  
             CPA                                     2.684 $\pm$ 2.109  
             scGen                          \textbf{2.600 $\pm$ 1.742}  
             SpatialProp                             9.446 $\pm$ 4.173  
             MintFlow                               15.132 $\pm$ 2.062  
             cellina (ablated)                       2.744 $\pm$ 2.218  
             cellina-GAT                             2.819 $\pm$ 2.101  
             cellina                                 3.031 $\pm$ 2.818  
             cellina$_{node-pert=200}$              13.696 $\pm$ 2.596  
             SpatialProp$_{node-pert=200}$          10.479 $\pm$ 4.535

In [24]:
def insert_midrule_perts(latex, table):
    groups = table.index.get_level_values(0)
    boundaries = [i for i in range(1, len(groups)) if groups[i] != groups[i - 1]]

    lines = latex.splitlines()
    new_lines = []

    row_idx = 0

    for line in lines:
        new_lines.append(line)

        # detect data rows (skip header)
        if "&" in line and "\\\\" in line:
            if row_idx in boundaries:
                new_lines.append(r"\midrule")
            row_idx += 1

    latex = "\n".join(new_lines)
    return latex

In [25]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
    latex = insert_midrule_perts(latex, table)
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")

print(latex)

(OUT_DIR / f"loo_summary_{DATASET_NAME}_DEG_{N_DEG}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 50 DEGs). Mean $\pm$ std across slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_crc}
\begin{tabular}{lccc}
\toprule
 & Pearson $\uparrow$ & Precision $\uparrow$ & E-dist (local) $\downarrow$ \\
\midrule
mean shift & 0.508 $\pm$ 0.255 & 0.180 $\pm$ 0.128 & 15.505 $\pm$ 4.520 \\
CPA & 0.680 $\pm$ 0.191 & 0.223 $\pm$ 0.188 & 2.684 $\pm$ 2.109 \\
scGen & 0.500 $\pm$ 0.384 & 0.216 $\pm$ 0.203 & \textbf{2.600 $\pm$ 1.742} \\
SpatialProp & 0.384 $\pm$ 0.344 & 0.082 $\pm$ 0.152 & 9.446 $\pm$ 4.173 \\
MintFlow & 0.659 $\pm$ 0.318 & 0.258 $\pm$ 0.213 & 15.132 $\pm$ 2.062 \\
cellina (ablated) & 0.722 $\pm$ 0.242 & 0.293 $\pm$ 0.203 & 2.744 $\pm$ 2.218 \\
cellina-GAT & \textbf{0.886 $\pm$ 0.093} & \textbf{0.409 $\pm$ 0.201} & 2.819 $\pm$ 2.101 \\
cellina & 0.847 $\pm$ 0.157 & 0.369 $\pm$ 0.197 & 3.031 $\pm$ 2.818 \\
cellina$_{node-pert=200}$ & 0.610 $\pm$ 0.302 & 0.273 $\pm$ 0.226 & 13.696 $\pm$ 2.596 \\

1133

In [25]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    #"rmse": "RMSE ↓",
    "avg_rank": "Avg. Rank ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method                       | Pearson ↑     | Precision ↑   | E-dist (local) ↓   | Avg. Rank ↓   |
|:-----------------------------|:--------------|:--------------|:-------------------|:--------------|
| ('CRC', 'mean shift')        | 0.508 ± 0.255 | 0.180 ± 0.128 | 15.505 ± 4.520     | 6.183 ± 1.142 |
| ('CRC', 'CPA')               | 0.680 ± 0.191 | 0.223 ± 0.188 | 2.684 ± 2.109      | 3.994 ± 0.660 |
| ('CRC', 'scGen')             | 0.500 ± 0.384 | 0.216 ± 0.203 | 2.600 ± 1.742      | 4.513 ± 1.309 |
| ('CRC', 'SpatialProp')       | 0.384 ± 0.344 | 0.082 ± 0.152 | 9.446 ± 4.173      | 6.633 ± 0.823 |
| ('CRC', 'MintFlow')          | 0.659 ± 0.318 | 0.258 ± 0.213 | 15.132 ± 2.062     | 5.473 ± 1.244 |
| ('CRC', 'cellina (ablated)') | 0.722 ± 0.242 | 0.293 ± 0.203 | 2.744 ± 2.218      | 3.300 ± 1.149 |
| ('CRC', 'cellina (gat)')     | 0.886 ± 0.093 | 0.409 ± 0.201 | 2.819 ± 2.101      | 2.406 ± 0.631 |
| ('CRC', 'cellina')           | 0.847 ± 0.157 | 0.369 ± 0.197 | 3.031 ± 2.818    

***